# 22.7 — Monte Carlo Tree Search

MCTS uses rollouts to spend search where reward and uncertainty are both high.

MCTS keeps the tree partial and estimates values by simulation. Its UCB score allocates frontier effort to actions with high reward estimates or limited evidence.

Save a copy to Drive to edit.

In [ ]:
import math
import random
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt

random.seed(22)


@dataclass
class StochasticTree:
    name: str
    rewards: Dict[Tuple[str, ...], float]
    actions: List[str]
    depth: int

    def legal_actions(self, state):
        if len(state) >= self.depth:
            return []
        return list(self.actions)

    def step(self, state, action, rng):
        next_state = tuple(list(state) + [action])
        done = len(next_state) >= self.depth
        reward = 0.0
        if done:
            mean = self.rewards.get(next_state, 0.0)
            reward = 1.0 if rng.random() < mean else 0.0
        return next_state, reward, done


@dataclass
class SearchNode:
    state: Tuple[str, ...]
    parent: Optional["SearchNode"] = None
    action: Optional[str] = None
    children: Dict[str, "SearchNode"] = field(default_factory=dict)
    visits: int = 0
    wins: float = 0.0


def ucb_score(wins, visits, parent_visits, c):
    if visits == 0:
        return math.inf
    return wins / visits + c * math.sqrt(math.log(parent_visits) / visits)


def select_child(node, c):
    return max(node.children.values(), key=lambda child: ucb_score(child.wins, child.visits, max(node.visits, 1), c))


def rollout(problem, state, rng):
    current = state
    total = 0.0
    done = False
    while not done:
        actions = problem.legal_actions(current)
        if not actions:
            break
        action = rng.choice(actions)
        current, reward, done = problem.step(current, action, rng)
        total += reward
    return total


def mcts(problem, rollouts=100, c=1.4, seed=22):
    rng = random.Random(seed)
    root = SearchNode(())
    expanded = 1
    for _ in range(rollouts):
        node = root
        state = node.state
        path = [node]
        while node.children and len(node.children) == len(problem.legal_actions(state)):
            node = select_child(node, c)
            state = node.state
            path.append(node)
        actions = problem.legal_actions(state)
        if actions:
            untried = [action for action in actions if action not in node.children]
            if untried:
                action = rng.choice(untried)
                next_state, reward, done = problem.step(state, action, rng)
                child = SearchNode(next_state, node, action)
                node.children[action] = child
                node = child
                state = next_state
                path.append(node)
                expanded += 1
                value = reward if done else reward + rollout(problem, state, rng)
            else:
                value = rollout(problem, state, rng)
        else:
            value = 0.0
        for item in path:
            item.visits += 1
            item.wins += value
    best = max(root.children.values(), key=lambda child: child.visits) if root.children else None
    return root, best, expanded


def make_tree(name, depth, actions, best_action, gap=0.25, sparse=False):
    rewards = {}
    for combo_index in range(len(actions) ** depth):
        digits = []
        x = combo_index
        for _ in range(depth):
            digits.append(actions[x % len(actions)])
            x = x // len(actions)
        state = tuple(digits)
        base = 0.35
        if state[0] == best_action:
            base += gap
        if sparse and state[-1] == actions[-1]:
            base += 0.15
        rewards[state] = min(base, 0.95)
    return StochasticTree(name, rewards, actions, depth)


def mcts_ladder():
    return [
        make_tree("D1 three-child trace", 1, ["A", "B", "C"], "B", 0.2),
        make_tree("D2 wider shallow", 2, ["A", "B", "C", "D"], "C", 0.2),
        make_tree("D3 deeper stochastic", 3, ["A", "B", "C"], "A", 0.18),
        make_tree("D4 sparse reward", 4, ["A", "B", "C"], "C", 0.12, True),
        make_tree("D5 deceptive exploration", 5, ["A", "B", "C", "D"], "D", 0.1, True),
    ]


def best_root_action(problem):
    totals = {}
    counts = {}
    for state, reward in problem.rewards.items():
        action = state[0]
        totals[action] = totals.get(action, 0.0) + reward
        counts[action] = counts.get(action, 0) + 1
    means = {action: totals[action] / counts[action] for action in totals}
    return max(means, key=means.get), means

## The concept, built once (D1)

MCTS selection uses $\text{UCB}_i=\bar X_i+c\sqrt{\frac{\ln N}{n_i}}$. The lesson uses $N=20$, visits $(10,2,8)$, wins $(6,1,5)$, and $c=1.4$.

In [ ]:
N_parent = 20
visits = {"A": 10, "B": 2, "C": 8}
wins = {"A": 6, "B": 1, "C": 5}
scores = {name: ucb_score(wins[name], visits[name], N_parent, 1.4) for name in visits}
rounded = {name: round(value, 3) for name, value in scores.items()}
print(rounded)
assert rounded["A"] == 1.366
assert rounded["B"] == 2.213
assert rounded["C"] == 1.482
assert max(scores, key=scores.get) == "B"

After a successful rollout on child $B$, visits go from $2$ to $3$, wins from $1$ to $2$, and the mean becomes $2/3=0.667$. Visit share is often the robust final action rule.

In [ ]:
new_mean = (1 + 1) / (2 + 1)
visit_share = 47 / 103
print("backup mean", round(new_mean, 3))
print("visit share", round(visit_share, 3))
assert round(new_mean, 3) == 0.667
assert round(visit_share, 3) == 0.456

## The dataset ladder

D1–D5 are seeded stochastic trees with wider actions, deeper rollouts, sparse rewards, and a deceptive low-exploration case.

In [ ]:
ladder = mcts_ladder()
for problem in ladder:
    best_action, means = best_root_action(problem)
    print(problem.name, "depth", problem.depth, "actions", len(problem.actions), "best", best_action, "means", means)

## Run the SAME method across D1–D5

Metric: action regret against known expected root means, plus visits/nodes expanded.

In [ ]:
results = []
for index, problem in enumerate(ladder):
    root, best, expanded = mcts(problem, rollouts=80 + index * 40, c=1.4, seed=22)
    optimal_action, means = best_root_action(problem)
    chosen = best.action
    regret = means[optimal_action] - means[chosen]
    results.append({"name": problem.name, "chosen": chosen, "optimal": optimal_action, "regret": regret, "expanded": expanded, "visits": root.visits})
print("rung | chosen | optimal | regret | expanded | visits")
for row in results:
    print(row["name"], row["chosen"], row["optimal"], round(row["regret"], 3), row["expanded"], row["visits"])

## Results visualization

Panels show MCTS tree growth by visits and expanded nodes. The curve shows regret and expansion cost versus rung size.

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(16, 3))
for axis, row in zip(axes, results):
    axis.bar(["visits", "expanded"], [row["visits"], row["expanded"]], color=["seagreen", "slateblue"])
    axis.set_title(row["name"].split()[0])
plt.show()
xs = list(range(1, len(results) + 1))
fig, axis = plt.subplots(figsize=(6, 3))
axis.plot(xs, [row["regret"] for row in results], marker="o", label="regret")
axis.plot(xs, [row["expanded"] for row in results], marker="s", label="expanded")
axis.legend()
axis.set_xlabel("D rung")
plt.show()

## Pitfall on D5: too little exploration

A small $c$ can lock onto lucky early rollouts. The fix is the lesson UCB exploration bonus and enough rollouts for sparse rewards.

In [ ]:
problem = ladder[-1]
root_low, best_low, expanded_low = mcts(problem, rollouts=120, c=0.05, seed=22)
root_fix, best_fix, expanded_fix = mcts(problem, rollouts=240, c=1.4, seed=22)
optimal, means = best_root_action(problem)
low_regret = means[optimal] - means[best_low.action]
fix_regret = means[optimal] - means[best_fix.action]
print("low exploration", best_low.action, round(low_regret, 3), expanded_low)
print("fixed exploration", best_fix.action, round(fix_regret, 3), expanded_fix)
assert fix_regret <= low_regret

## Evaluate it + Practice

- Metric: regret and expanded tree nodes.
- No-skill baseline: choose by first lucky empirical mean.
- Cheap sanity check: D1 UCB values round to 1.366, 2.213, 1.482.
- Ablation: set c near zero and regret rises on D5.
- Failure signals: chosen action has low visits or high regret after more rollouts.

### Practice
Change c across a grid and plot D5 regret.

### Practice
Choose final action by mean instead of visits and compare variance.

### Practice
Make D4 rewards sparser and increase rollouts.